In [ ]:
import os
ssd_path = "/Volumes/samsung_nvme/.cache/huggingface"

if os.path.exists("/Volumes/samsung_nvme"):
    os.makedirs(ssd_path, exist_ok=True)
    os.environ["HF_HOME"] = ssd_path
    print(f"HF_HOME impostato su: {os.environ['HF_HOME']}")
else:
    os.unsetenv("HF_HOME")
    print("ATTENZIONE: SSD non trovato, Hugging Face userà il percorso di default.")

In [ ]:
!echo $HF_HOME

In [42]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

#repo_id = "lerobot/toto"
repo_id = "lerobot/pusht"
dataset = LeRobotDataset(repo_id)

In [ ]:
# print all the attributes of LeRobotDataset object
[att for att in dir(dataset) if not att.startswith("__")]

## Memory analysis

In [55]:
import pandas as pd

data_dir = dataset.root / "data"
parquet_files = sorted(data_dir.glob("*/*.parquet"))
df = pd.read_parquet(f"{parquet_files[0]}").reset_index(drop=True) # manually laod first parquet data file

In [13]:
import numpy as np
len_first_file = len(np.array(dataset.meta.episodes["data/file_index"])[np.array(dataset.meta.episodes["data/file_index"])==0]) # episodes stored in first parquet data file


In [ ]:
cum_mb = 0
for i in range(len_first_file):
    dfi = dataset.meta.episodes["dataset_from_index"][i]
    dti = dataset.meta.episodes["dataset_to_index"][i]
    size_mb = df.iloc[dfi:dti].memory_usage(deep=True).sum() / (1024 * 1024)
    #print(f"size = {size_mb} MB")
    cum_mb += size_mb

print(f"cum size = {cum_mb} MB")

cum size = 92.87190818786621 MB


In [79]:
type(df["action0"].iloc[0])

numpy.ndarray

In [78]:
df.iloc[0:1].memory_usage(deep=True).sum() #/ (10 * 1024 * 1024) #* 25650 / 1024

np.int64(80414)

In [18]:
df.iloc[1:2]["action0"][1].nbytes

80000

In [26]:
tot_freames = dataset.meta.episodes[-1]["dataset_to_index"]
tot_freames

25650

In [52]:
dataset.hf_dataset[-1]["action0"]

tensor([[0.4161, 0.7114, 0.4239,  ..., 0.9050, 0.2292, 0.3413],
        [0.1570, 0.2190, 0.3251,  ..., 0.0219, 0.1028, 0.1171],
        [0.8496, 0.4987, 0.4921,  ..., 0.6442, 0.6787, 0.4194],
        ...,
        [0.5513, 0.8176, 0.7310,  ..., 0.8922, 0.4079, 0.6504],
        [0.7882, 0.3475, 0.3561,  ..., 0.3231, 0.8779, 0.2487],
        [0.3643, 0.9283, 0.4326,  ..., 0.7555, 0.0536, 0.7068]])

## Video analysis

In [27]:
dataset.vcodec

'libsvtav1'

In [43]:
dataset.meta.video_keys

['observation.image']

In [ ]:
original_video_key = "observation.images.image"
new_video_key = "observation.image.test"

In [ ]:
for idx, ep in enumerate(dataset.meta.episodes):
    if ep[f"videos/{new_video_key}/file_index"] > ep[f"videos/{original_video_key}/file_index"]:
        print(f"{idx} > test")
    elif ep[f"videos/{new_video_key}/file_index"] < ep[f"videos/{original_video_key}/file_index"]:
        print(f"{idx} > image")

In [ ]:
print(dataset.meta.episodes[f"videos/{new_video_key}/chunk_index"]==dataset.meta.episodes[f"videos/{original_video_key}/chunk_index"])
print(dataset.meta.episodes[f"videos/{new_video_key}/file_index"]==dataset.meta.episodes[f"videos/{original_video_key}/file_index"])

In [ ]:
print(dataset.meta.episodes[f"videos/{new_video_key}/from_timestamp"]==dataset.meta.episodes[f"videos/{original_video_key}/from_timestamp"])
print(dataset.meta.episodes[f"videos/{new_video_key}/to_timestamp"]==dataset.meta.episodes[f"videos/{original_video_key}/to_timestamp"])

In [ ]:
print(dataset.meta.episodes[f"videos/{new_video_key}/from_timestamp"])
print(dataset.meta.episodes[f"videos/{new_video_key}/to_timestamp"])

In [ ]:
print(dataset.meta.episodes[f"videos/{original_video_key}/from_timestamp"])
print(dataset.meta.episodes[f"videos/{original_video_key}/to_timestamp"])

In [ ]:
dataset.meta.info["features"][original_video_key]

In [ ]:
from lerobot.datasets.video_utils import get_video_info
video_info = get_video_info("/Users/claudiomacaluso/Documents/code/lerobot_contribution/file-000.mp4")
shape = (
    video_info.pop("video.height"),
    video_info.pop("video.width"),
    video_info.pop("video.channels")
)
new_video_info = {
    "dtype": "video",
    "shape": shape,
    "names": ["height", "width", "channels"],
    "video_info": video_info
} # then change fps with target one

In [ ]:
new_video_info

## Data analysis

In [ ]:
# make a fake data filled with ones matching the total episodes in order to test add_feature
import numpy as np
import pickle

arr = np.random.rand(25650, 1000, 10) # pusht compliant dimensions

In [ ]:
arr.nbytes/(1024*1024*1024) # 1.9 GB

In [ ]:

with open('data/pusht_test.pkl', 'wb') as f:
    pickle.dump(arr, f)

In [ ]:
import pickle

with open('data/pusht_test.pkl', 'rb') as f:
    arr = pickle.load(f)

print(arr.nbytes/(1024*1024*1024))
print(arr.shape)

## Metadata analysis

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata

meta = LeRobotDatasetMetadata("lerobot/pusht", "/Users/claudiomacaluso/.cache/huggingface/lerobot/lerobot/pusht", "v3.0", force_cache_sync=False) # manually loading metadata

In [35]:
dataset.meta.features

{'observation.image': {'dtype': 'video',
  'shape': (96, 96, 3),
  'names': ['height', 'width', 'channel'],
  'video_info': {'video.fps': 10.0,
   'video.codec': 'av1',
   'video.pix_fmt': 'yuv420p',
   'video.is_depth_map': False,
   'has_audio': False},
  'info': {}},
 'observation.state': {'dtype': 'float32',
  'shape': (2,),
  'names': {'motors': ['motor_0', 'motor_1']},
  'fps': 10.0},
 'action': {'dtype': 'float32',
  'shape': (2,),
  'names': {'motors': ['motor_0', 'motor_1']},
  'fps': 10.0},
 'episode_index': {'dtype': 'int64', 'shape': (1,), 'names': None},
 'frame_index': {'dtype': 'int64', 'shape': (1,), 'names': None},
 'timestamp': {'dtype': 'float32', 'shape': (1,), 'names': None},
 'next.reward': {'dtype': 'float32',
  'shape': (1,),
  'names': None,
  'fps': 10.0},
 'next.done': {'dtype': 'bool', 'shape': (1,), 'names': None, 'fps': 10.0},
 'next.success': {'dtype': 'bool', 'shape': (1,), 'names': None, 'fps': 10.0},
 'index': {'dtype': 'int64', 'shape': (1,), 'names':

In [38]:
dataset.hf_dataset[0]

{'observation.state': tensor([222.,  97.]),
 'action': tensor([233.,  71.]),
 'episode_index': tensor(0),
 'frame_index': tensor(0),
 'timestamp': tensor(0.),
 'next.reward': tensor(0.1903),
 'next.done': tensor(False),
 'next.success': tensor(False),
 'index': tensor(0),
 'task_index': tensor(0),
 'action0': tensor([[0.9115, 0.2460, 0.0472,  ..., 0.5735, 0.5920, 0.1976],
         [0.0832, 0.0849, 0.6110,  ..., 0.7702, 0.7654, 0.6602],
         [0.2753, 0.9434, 0.8826,  ..., 0.8005, 0.5359, 0.5370],
         ...,
         [0.4919, 0.1994, 0.0361,  ..., 0.4962, 0.3918, 0.6500],
         [0.9330, 0.1250, 0.9733,  ..., 0.7871, 0.7148, 0.3615],
         [0.8317, 0.9898, 0.5297,  ..., 0.7530, 0.9069, 0.7641]])}

In [37]:
dataset.meta.episodes[0]

{'episode_index': 0,
 'data/chunk_index': 0,
 'data/file_index': 0,
 'dataset_from_index': 0,
 'dataset_to_index': 161,
 'videos/observation.image/chunk_index': 0,
 'videos/observation.image/file_index': 0,
 'videos/observation.image/from_timestamp': 0.0,
 'videos/observation.image/to_timestamp': 16.1,
 'tasks': ['Push the T-shaped block onto the T-shaped target.'],
 'length': 161,
 'meta/episodes/chunk_index': 0,
 'meta/episodes/file_index': 0}

In [ ]:
dataset.meta.episodes.features

datasets.features.features.Features

In [39]:
dataset.meta.video_path #.format()

'videos/{video_key}/chunk-{chunk_index:03d}/file-{file_index:03d}.mp4'